# Customer Churn Analysis
**Virtual Work Lab — Data Analytics Internship | Task 2**

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from churn_utils import load_data, engineer_features, churn_rate_by_group, top_churn_signals
from churn_utils import plot_churn_trend, plot_churn_by_tier, plot_churn_drivers

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully')

## 1. Load & Explore Data

In [ ]:
# Load your dataset
# df = load_data('../data/raw/customers.csv')

# --- Sample data for demonstration ---
np.random.seed(42)
n = 5000
df = pd.DataFrame({
    'customer_id': range(1, n+1),
    'subscription_tier': np.random.choice(['Basic','Pro','Enterprise'], n, p=[0.6,0.3,0.1]),
    'signup_date': pd.date_range('2025-01-01', periods=n, freq='2h'),
    'last_login_date': pd.Timestamp.today() - pd.to_timedelta(np.random.exponential(10, n), unit='d'),
    'weekly_feature_uses': np.random.poisson(4, n),
    'support_tickets': np.random.poisson(0.8, n),
    'onboarding_completion_pct': np.random.uniform(0, 100, n),
    'team_size': np.random.choice([1,2,3,4,5,10,20], n, p=[0.4,0.2,0.15,0.1,0.07,0.05,0.03]),
    'integrations_connected': np.random.choice([0,1,2,3], n, p=[0.55,0.25,0.15,0.05]),
})
df['churned'] = ((df['weekly_feature_uses'] < 3) & (df['onboarding_completion_pct'] < 50)).astype(int)

print(f'Dataset: {df.shape[0]:,} customers, {df.shape[1]} columns')
df.head()

## 2. Feature Engineering

In [ ]:
df = engineer_features(df)
print('Overall churn rate:', f"{df['churned'].mean()*100:.1f}%")
df[['days_since_login','tenure_months','churn_risk_score']].describe().round(2)

## 3. Churn by Tier

In [ ]:
tier_rates = churn_rate_by_group(df, 'subscription_tier')
print(tier_rates)
fig = plot_churn_by_tier(df, save_path='../assets/charts/churn_by_tier.png')
plt.show()

## 4. Behavioral Churn Drivers

In [ ]:
signals = top_churn_signals(df)
print(signals)
fig = plot_churn_drivers(df, save_path='../assets/charts/churn_drivers.png')
plt.show()

## 5. At-Risk Segment Summary

In [ ]:
high_risk = df[df['churn_risk_score'] >= 0.5]
print(f'High-risk users: {len(high_risk):,} ({len(high_risk)/len(df)*100:.1f}% of base)')
print()
print('Risk distribution by tier:')
print(df.groupby('subscription_tier')['churn_risk_score'].mean().round(3))

## 6. Conclusions

See `reports/churn_analysis_report.md` for the full written analysis and recommendations.